In [2]:
import pandas as pd
import os

'''a = np.zeros((500, 10))

for i in tqdm(range(5), desc='Training 5 XGBoost for Ensembling'):
    model = TabPFNStackingPipeline(meta_model=XGBRegressor)
    output = model.get_submission(path=None, only_preds=True)
    a += output


sub = pd.DataFrame(data= range(1, 501), columns=['ID'])
sub[[f'BlendProperty{i+1}' for i in range(10)]] = a
sub.to_csv(os.path.join('submissions', 'tab_xgb_ensemble_5.csv'), index=False)'''

"a = np.zeros((500, 10))\n\nfor i in tqdm(range(5), desc='Training 5 XGBoost for Ensembling'):\n    model = TabPFNStackingPipeline(meta_model=XGBRegressor)\n    output = model.get_submission(path=None, only_preds=True)\n    a += output\n\n\nsub = pd.DataFrame(data= range(1, 501), columns=['ID'])\nsub[[f'BlendProperty{i+1}' for i in range(10)]] = a\nsub.to_csv(os.path.join('submissions', 'tab_xgb_ensemble_5.csv'), index=False)"

In [21]:
a = pd.read_csv(r"C:\Users\LENOVO\Downloads\tab_xgb_ensemble_10 (1).csv").drop(columns=['ID']).to_numpy()
b = pd.read_csv(r"C:\Users\LENOVO\Downloads\tab_xgb_ensemble_10.csv").drop(columns=['ID']).to_numpy() / 10
c = pd.read_csv(r"C:\Users\LENOVO\Downloads\tab_xgb_ensemble_5 (1).csv").drop(columns=['ID']).to_numpy() / 10
d = pd.read_csv(r'submissions\tabpfn_quantile_xgb.csv').drop(columns=['ID']).to_numpy() 
e = pd.read_csv(r"C:\Users\LENOVO\Downloads\tab_xgb_ensemble_20.csv").drop(columns=['ID']).to_numpy()

f = (a + b + c + d + e)/5

In [22]:
sub = pd.DataFrame(data= range(1, 501), columns=['ID'])
sub[[f'BlendProperty{i+1}' for i in range(10)]] = f
sub.to_csv(os.path.join('submissions', 'tab_xgb_ensemble_46.csv'), index=False)

In [20]:
d

array([[ 0.013383  ,  0.02782939,  0.0779702 , ...,  0.02030416,
        -0.03517869,  0.03131644],
       [-0.07113199, -0.07782626, -0.11764345, ..., -0.10986795,
        -0.08824616, -0.00312161],
       [ 0.17887689,  0.1173273 ,  0.11210532, ...,  0.20337918,
         0.07205207,  0.22232828],
       ...,
       [ 0.19270009,  0.20256631,  0.0315957 , ...,  0.11514281,
         0.02185993,  0.04367588],
       [-0.01928211,  0.08460024,  0.16291862, ...,  0.04462186,
         0.00402199,  0.12221953],
       [-0.11052213, -0.19539953, -0.19488939, ..., -0.19042134,
        -0.19707543, -0.0232242 ]])

In [1]:
from ingestion import DataFrameLoader

loader = DataFrameLoader()
x, _, _ = loader.load(split_labels=True, reduce_dims=True)

Loading data...
Reducing Feature Dimnesions
Reducing Feature Dimnesions
Shape of feature matrix is (2000, 15)
Shape of label matrix is (2000, 10)
Shape of Test Data is (500, 15)


In [3]:
x[0]

array([ 0.21      ,  0.        ,  0.42      ,  0.25      ,  0.12      ,
       -0.08903126,  0.01753703, -0.01815405,  0.19912644, -0.05705431,
       -0.05075045,  0.05560501,  0.00370947,  0.01765581, -0.01526547])

In [5]:
_[0]

array([ 0.18      ,  0.05      ,  0.32      ,  0.37      ,  0.08      ,
       -0.06024945,  0.12220264, -0.04917761,  0.01078063,  0.03281893,
       -0.03558709,  0.09199725, -0.03491649, -0.04851419,  0.04244394])

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt
import seaborn as sns

def engineer_fuel_blend_features(df):
    """
    Engineer features for fuel blend composition and properties prediction
    
    Args:
        df: DataFrame with columns as described (5 fractions + 50 component properties + 10 targets)
    
    Returns:
        DataFrame with original features plus engineered features
    """
    
    # Create a copy to avoid modifying original data
    df_engineered = df.copy()
    
    # Dictionary to store all new features - more efficient than repeated insertion
    new_features = {}
    
    # Define column groups
    fraction_cols = [f'Component{i}_fraction' for i in range(1, 6)]
    
    # Component properties organized by property type
    property_cols = {}
    for prop_num in range(1, 11):
        property_cols[f'Property{prop_num}'] = [f'Component{i}_Property{prop_num}' for i in range(1, 6)]
    
    # =============================================================================
    # 1. FRACTION-BASED FEATURES
    # =============================================================================
    
    # Basic fraction statistics
    new_features['total_fraction'] = df_engineered[fraction_cols].sum(axis=1)
    new_features['fraction_diversity'] = df_engineered[fraction_cols].apply(
        lambda x: -np.sum(x * np.log(x + 1e-10)), axis=1)  # Shannon entropy
    
    # FIXED: Convert dominant component to numeric index
    dominant_indices = df_engineered[fraction_cols].idxmax(axis=1)
    new_features['dominant_component'] = dominant_indices.str.extract(r'Component(\d+)').astype(int).iloc[:, 0]
    
    new_features['max_fraction'] = df_engineered[fraction_cols].max(axis=1)
    new_features['min_fraction'] = df_engineered[fraction_cols].min(axis=1)
    new_features['fraction_std'] = df_engineered[fraction_cols].std(axis=1)
    new_features['fraction_range'] = new_features['max_fraction'] - new_features['min_fraction']
    
    # Effective number of components (inverse Simpson index)
    new_features['effective_components'] = 1 / (df_engineered[fraction_cols] ** 2).sum(axis=1)
    
    # Binary indicators for component presence
    for i, col in enumerate(fraction_cols, 1):
        new_features[f'has_component_{i}'] = (df_engineered[col] > 0.01).astype(int)
    
    new_features['num_active_components'] = sum(new_features[f'has_component_{i}'] for i in range(1, 6))
    
    # =============================================================================
    # 2. WEIGHTED PROPERTY FEATURES (Volume-weighted averages)
    # =============================================================================
    
    for prop_name, prop_cols in property_cols.items():
        # Volume-weighted average of each property
        weighted_sum = sum(df_engineered[fraction_cols[i]] * df_engineered[prop_cols[i]] 
                          for i in range(5))
        new_features[f'weighted_avg_{prop_name}'] = weighted_sum
        
        # Weighted standard deviation
        weighted_mean = weighted_sum
        weighted_var = sum(df_engineered[fraction_cols[i]] * 
                          (df_engineered[prop_cols[i]] - weighted_mean) ** 2 
                          for i in range(5))
        new_features[f'weighted_std_{prop_name}'] = np.sqrt(weighted_var)
        
        # Property range across components
        new_features[f'range_{prop_name}'] = (df_engineered[prop_cols].max(axis=1) - 
                                               df_engineered[prop_cols].min(axis=1))
        
        # Coefficient of variation
        new_features[f'cv_{prop_name}'] = (df_engineered[prop_cols].std(axis=1) / 
                                           (df_engineered[prop_cols].mean(axis=1) + 1e-10))
    
    # =============================================================================
    # 3. COMPONENT-SPECIFIC FEATURES
    # =============================================================================
    
    # For each component, create aggregate features
    for comp_num in range(1, 6):
        comp_prop_cols = [f'Component{comp_num}_Property{prop}' for prop in range(1, 11)]
        
        # Component property statistics
        new_features[f'Component{comp_num}_prop_mean'] = df_engineered[comp_prop_cols].mean(axis=1)
        new_features[f'Component{comp_num}_prop_std'] = df_engineered[comp_prop_cols].std(axis=1)
        new_features[f'Component{comp_num}_prop_sum'] = df_engineered[comp_prop_cols].sum(axis=1)
        
        # Weighted contribution (fraction * property characteristics)
        new_features[f'Component{comp_num}_contribution'] = (
            df_engineered[f'Component{comp_num}_fraction'] * 
            new_features[f'Component{comp_num}_prop_mean']
        )
    
    # =============================================================================
    # 4. INTERACTION FEATURES
    # =============================================================================
    
    # Pairwise component interactions
    for i in range(1, 6):
        for j in range(i+1, 6):
            # Fraction interaction
            new_features[f'fraction_interaction_{i}_{j}'] = (
                df_engineered[f'Component{i}_fraction'] * 
                df_engineered[f'Component{j}_fraction']
            )
            
            # Property similarity (for each property type)
            for prop_num in range(1, 11):
                prop_diff = abs(df_engineered[f'Component{i}_Property{prop_num}'] - 
                               df_engineered[f'Component{j}_Property{prop_num}'])
                new_features[f'prop{prop_num}_diff_{i}_{j}'] = prop_diff
                new_features[f'prop{prop_num}_avg_{i}_{j}'] = (
                    df_engineered[f'Component{i}_Property{prop_num}'] + 
                    df_engineered[f'Component{j}_Property{prop_num}']
                ) / 2
    
    # =============================================================================
    # 6. DIMENSIONALITY REDUCTION FEATURES
    # =============================================================================
    
    # PCA on component properties
    all_component_props = []
    for comp_num in range(1, 6):
        comp_props = [f'Component{comp_num}_Property{prop}' for prop in range(1, 11)]
        all_component_props.extend(comp_props)
    
    # Apply PCA to reduce dimensionality while preserving variance
    pca = PCA(n_components=10)  # Reduce 50 features to 10 principal components
    pca_features = pca.fit_transform(df_engineered[all_component_props])
    
    for i in range(10):
        new_features[f'component_pca_{i+1}'] = pca_features[:, i]
    
    # =============================================================================
    # 7. BLEND COMPLEXITY FEATURES
    # =============================================================================
    
    # Measure how "complex" the blend is
    new_features['blend_complexity_score'] = (
        new_features['fraction_diversity'] * 
        new_features['num_active_components'] * 
        np.mean([new_features[f'weighted_std_Property{i}'] for i in range(1, 11)], axis=0)
    )
    
    # Homogeneity index (how similar are the components)
    property_similarities = []
    for prop_num in range(1, 11):
        prop_cols = [f'Component{i}_Property{prop_num}' for i in range(1, 6)]
        prop_std = df_engineered[prop_cols].std(axis=1)
        property_similarities.append(prop_std)
    
    new_features['blend_homogeneity'] = 1 / (1 + np.mean(property_similarities, axis=0))
    
    # =============================================================================
    # 8. RATIO FEATURES
    # =============================================================================
    
    # Key property ratios that might be important for fuel performance
    for prop1 in range(1, 11):
        for prop2 in range(prop1+1, 11):
            ratio_name = f'weighted_ratio_Property{prop1}_Property{prop2}'
            prop1_weighted = new_features[f'weighted_avg_Property{prop1}']
            prop2_weighted = new_features[f'weighted_avg_Property{prop2}']
            new_features[ratio_name] = prop1_weighted / (prop2_weighted + 1e-10)
    
    # =============================================================================
    # 9. STABILITY INDICATORS
    # =============================================================================
    
    # Measures that might indicate blend stability
    for prop_num in range(1, 11):
        prop_cols = [f'Component{i}_Property{prop_num}' for i in range(1, 6)]
        
        # Weighted coefficient of variation
        weighted_mean = new_features[f'weighted_avg_Property{prop_num}']
        weighted_cv = new_features[f'weighted_std_Property{prop_num}'] / (weighted_mean + 1e-10)
        new_features[f'stability_index_Property{prop_num}'] = 1 / (1 + weighted_cv)
    
    # =============================================================================
    # 10. DOMINANT COMPONENT FEATURES
    # =============================================================================
    
    # Features based on the most abundant component
    dominant_comp_idx = df_engineered[fraction_cols].idxmax(axis=1)
    
    for prop_num in range(1, 11):
        dominant_prop_values = []
        for idx, dominant_comp in enumerate(dominant_comp_idx):
            # Extract component number from column name like 'Component1_fraction'
            comp_number = dominant_comp.split('Component')[1].split('_')[0]
            prop_col = f'Component{comp_number}_Property{prop_num}'
            dominant_prop_values.append(df_engineered.iloc[idx][prop_col])
        
        new_features[f'dominant_component_Property{prop_num}'] = dominant_prop_values
    
    # =============================================================================
    # COMBINE ALL FEATURES
    # =============================================================================
    
    # Convert new_features dict to DataFrame and concatenate with original
    new_features_df = pd.DataFrame(new_features, index=df_engineered.index)
    df_engineered = pd.concat([df_engineered, new_features_df], axis=1)
    
    return df_engineered

def analyze_feature_importance(df_engineered, target_columns=None, top_n=20):
    """
    Comprehensive feature importance analysis with group-wise insights
    """
    if target_columns is None:
        target_columns = [f'BlendProperty{i}' for i in range(1, 11)]
    
    # Separate features and targets
    feature_cols = [col for col in df_engineered.columns if not col.startswith('BlendProperty')]
    
    # Define feature groups
    feature_groups = {
        'original_fractions': [col for col in feature_cols if col.endswith('_fraction')],
        'original_properties': [col for col in feature_cols if '_Property' in col and 'weighted' not in col and 'range' not in col and 'cv' not in col and 'contribution' not in col and 'dominant' not in col],
        
        'fraction_based': [col for col in feature_cols if any(x in col for x in ['fraction_diversity', 'dominant_component', 'max_fraction', 'min_fraction', 'fraction_std', 'fraction_range', 'effective_components', 'has_component', 'num_active_components', 'total_fraction'])],
        
        'weighted_properties': [col for col in feature_cols if col.startswith('weighted_avg_') or col.startswith('weighted_std_')],
        
        'component_aggregates': [col for col in feature_cols if any(x in col for x in ['_prop_mean', '_prop_std', '_prop_sum', '_contribution']) and 'Component' in col],
        
        'interactions': [col for col in feature_cols if any(x in col for x in ['interaction', '_diff_', '_avg_']) and any(str(i) in col for i in range(1, 6))],
        
        'synergy_antagonism': [col for col in feature_cols if 'contribution_pct' in col],
        
        'pca_features': [col for col in feature_cols if 'component_pca' in col],
        
        'complexity_blend': [col for col in feature_cols if any(x in col for x in ['blend_complexity', 'blend_homogeneity'])],
        
        'property_ranges': [col for col in feature_cols if col.startswith('range_') or col.startswith('cv_')],
        
        'ratios': [col for col in feature_cols if 'weighted_ratio' in col],
        
        'stability': [col for col in feature_cols if 'stability_index' in col],
        
        'dominant_component': [col for col in feature_cols if 'dominant_component_Property' in col]
    }
    
    print("="*80)
    print("FEATURE ENGINEERING ANALYSIS")
    print("="*80)
    
    print(f"Original features: {len([col for col in df_engineered.columns if col.endswith('_fraction') or ('_Property' in col and 'weighted' not in col and 'range' not in col and 'cv' not in col and 'contribution' not in col and 'dominant' not in col)])}")
    print(f"Engineered features: {len(feature_cols) - len([col for col in df_engineered.columns if col.endswith('_fraction') or ('_Property' in col and 'weighted' not in col and 'range' not in col and 'cv' not in col and 'contribution' not in col and 'dominant' not in col)])}")
    print(f"Total features: {len(feature_cols)}")
    print(f"Target variables: {len(target_columns)}")
    
    print("\nFeature Groups Summary:")
    print("-" * 50)
    for group, features in feature_groups.items():
        print(f"{group:25}: {len(features):4d} features")
    
    # Calculate feature importance for each target
    print("\n" + "="*80)
    print("FEATURE IMPORTANCE ANALYSIS")
    print("="*80)
    
    group_importance = {}
    overall_feature_importance = {}
    
    for target in target_columns:
        if target in df_engineered.columns:
            print(f"\nAnalyzing target: {target}")
            
            X = df_engineered[feature_cols]
            y = df_engineered[target]
            
            # Handle missing values
            X_clean = X.fillna(X.mean())
            y_clean = y.fillna(y.mean())
            
            # Calculate mutual information
            try:
                mi_scores = mutual_info_regression(X_clean, y_clean, random_state=42)
                feature_importance = dict(zip(feature_cols, mi_scores))
                overall_feature_importance[target] = feature_importance
                
                # Calculate group importance
                group_scores = {}
                for group, features in feature_groups.items():
                    if features:  # Only if group has features
                        group_score = np.mean([feature_importance.get(feat, 0) for feat in features])
                        group_scores[group] = group_score
                
                group_importance[target] = group_scores
                
                # Show top features for this target
                sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)
                print(f"Top {min(10, len(sorted_features))} features for {target}:")
                for i, (feat, score) in enumerate(sorted_features[:10]):
                    print(f"  {i+1:2d}. {feat:40s}: {score:.6f}")
                    
            except Exception as e:
                print(f"Error calculating importance for {target}: {e}")
    
    # Calculate average group importance across all targets
    print("\n" + "="*80)
    print("GROUP IMPORTANCE RANKING (Average across all targets)")
    print("="*80)
    
    if group_importance:
        avg_group_importance = {}
        for group in feature_groups.keys():
            scores = [group_importance[target].get(group, 0) for target in group_importance.keys()]
            avg_group_importance[group] = np.mean(scores)
        
        sorted_groups = sorted(avg_group_importance.items(), key=lambda x: x[1], reverse=True)
        
        print("Rank | Group Name                | Avg Score | Recommendation")
        print("-" * 70)
        for i, (group, score) in enumerate(sorted_groups):
            recommendation = "HIGH PRIORITY" if score > 0.1 else "MEDIUM PRIORITY" if score > 0.05 else "LOW PRIORITY"
            print(f"{i+1:4d} | {group:25s} | {score:9.6f} | {recommendation}")
    
    # Provide actionable insights
    print("\n" + "="*80)
    print("ACTIONABLE INSIGHTS & RECOMMENDATIONS")
    print("="*80)
    
    if group_importance:
        top_3_groups = sorted_groups[:3]
        print("🔥 TOP 3 MOST IMPORTANT FEATURE GROUPS:")
        for i, (group, score) in enumerate(top_3_groups):
            print(f"{i+1}. {group.upper()} (Score: {score:.6f})")
            
            # Get sample features from this group
            sample_features = feature_groups[group][:5]
            print(f"   Key features: {', '.join(sample_features)}")
            
            # Provide specific recommendations
            if group == 'weighted_properties':
                print("   💡 Focus on volume-weighted property averages - these capture the true blend characteristics")
            elif group == 'interactions':
                print("   💡 Component interactions are crucial - consider more pairwise features")
            elif group == 'fraction_based':
                print("   💡 Fraction statistics matter - blend composition diversity is key")
            elif group == 'component_aggregates':
                print("   💡 Individual component characteristics significantly impact blend properties")
            elif group == 'complexity_blend':
                print("   💡 Blend complexity metrics are important - consider more homogeneity measures")
            print()
    
    # Feature selection recommendations
    print("📋 FEATURE SELECTION RECOMMENDATIONS:")
    
    if overall_feature_importance:
        # Get top features across all targets
        all_scores = {}
        for target_scores in overall_feature_importance.values():
            for feat, score in target_scores.items():
                if feat not in all_scores:
                    all_scores[feat] = []
                all_scores[feat].append(score)
        
        # Average scores across targets
        avg_scores = {feat: np.mean(scores) for feat, scores in all_scores.items()}
        top_features = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
        
        print(f"• Keep TOP {top_n} features overall:")
        for i, (feat, score) in enumerate(top_features[:top_n]):
            print(f"  {i+1:2d}. {feat}")
        
        # Identify low-value features to remove
        low_value_features = [feat for feat, score in top_features if score < 0.01]
        print(f"\n• Consider removing {len(low_value_features)} low-value features (score < 0.01)")
        
        # Group-wise recommendations
        print(f"\n• RECOMMENDED FEATURE GROUPS TO PRIORITIZE:")
        for group, score in sorted_groups[:5]:
            print(f"  ✓ {group} (score: {score:.6f})")
    
    return {
        'feature_importance': overall_feature_importance,
        'group_importance': group_importance,
        'feature_groups': feature_groups,
        'avg_group_importance': avg_group_importance if 'avg_group_importance' in locals() else {},
        'top_features': top_features if 'top_features' in locals() else []
    }

# Example usage:
"""

"""

'\n\n'

In [2]:
# Load your data
df = pd.read_csv(r'dataset\train.csv')

# Engineer features
df_with_features = engineer_fuel_blend_features(df)

# Comprehensive analysis
target_columns = [f'BlendProperty{i}' for i in range(1, 11)]
results = analyze_feature_importance(df_with_features, target_columns)

# Access specific results
print("Top 5 most important feature groups:")
for group, score in list(results['avg_group_importance'].items())[:5]:
    print(f"{group}: {score:.6f}")

FEATURE ENGINEERING ANALYSIS
Original features: 68
Engineered features: 348
Total features: 416
Target variables: 10

Feature Groups Summary:
--------------------------------------------------
original_fractions       :    8 features
original_properties      :   60 features
fraction_based           :   24 features
weighted_properties      :   20 features
component_aggregates     :   20 features
interactions             :  216 features
synergy_antagonism       :    0 features
pca_features             :   10 features
complexity_blend         :    2 features
property_ranges          :   20 features
ratios                   :   45 features
stability                :   10 features
dominant_component       :   10 features

FEATURE IMPORTANCE ANALYSIS

Analyzing target: BlendProperty1
Top 10 features for BlendProperty1:
   1. fraction_interaction_4_5                : 0.292983
   2. Component5_fraction                     : 0.258175
   3. fraction_interaction_3_5                : 0.177330
   4

In [3]:
from ingestion import DataFrameLoader

loader = DataFrameLoader()


x, _, x_test = loader.load(split_labels=True)

Loading data...
Shape of feature matrix is (2000, 55)
Shape of label matrix is (2000, 10)
Shape of Test Data is (500, 55)


In [4]:
_[0]

array([ 0.51910126, -1.25769025,  1.78529868, ...,  0.06019995,
       -0.20454596, -1.82081881])

In [3]:
from tabpfn import TabPFNRegressor

model = TabPFNRegressor()

In [4]:
type(model) == TabPFNRegressor

True

In [5]:
isinstance(model, TabPFNRegressor)

True

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, LSTM

def build_model(input_shape=(16, 224, 224, 3), num_classes=6):
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    mobilenet.trainable = False
    
    base_cnn = tf.keras.layers.TimeDistributed(mobilenet)(inputs)
    
    x = tf.keras.layers.TimeDistributed(GlobalAveragePooling2D())(base_cnn)
    
    x = tf.keras.layers.Bidirectional(LSTM(64, return_sequences=True))(x)
    x = tf.keras.layers.Bidirectional(LSTM(32, return_sequences=False))(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(64, activation='relu', 
                             kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    
    pose_estimation = tf.keras.layers.Dense(num_classes, activation='softmax', name='pose_estimation')(x)
    pose_rating = tf.keras.layers.Dense(3, activation='softmax', name='pose_rating')(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=[pose_estimation, pose_rating])
    
    model.compile(optimizer='adam', 
                  loss={'pose_estimation': 'categorical_crossentropy',
                        'pose_rating': 'categorical_crossentropy'}, 
                  metrics={'pose_estimation': 'accuracy',
                           'pose_rating': 'accuracy'})
     
    return model

model = build_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 16, 224,   │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_4  │ (None, 16, 7, 7,  │  2,257,984 │ input_layer_6[0]… │
│ (TimeDistributed)   │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_5  │ (None, 16, 1280)  │          0 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 16, 128)   │    688,640 │ time_distributed… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 64)        │     41,216 │ bidirectional_1[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      4,160 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pose_estimation     │ (None, 6)         │        390 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pose_rating (Dense) │ (None, 3)         │        195 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,992,585 (11.42 MB)

 Trainable params: 734,601 (2.80 MB)

 Non-trainable params: 2,257,984 (8.61 MB)